### Cleaning & Preprocessing Summary

The code prepares the YouTube comments dataset (and later Reddit, if added) for sentiment, topic, and statistical analysis.  
It performs all data cleaning, feature extraction, filtering, and balancing in one step, and exports ready-to-analyze files.


## For Youtube

### ⚙️ Step-by-Step Breakdown for YT




#### 1️⃣ Setup & Imports
- Installs and imports `pandas`, `numpy`, `emoji`, and `langdetect`.
- Loads `data/comments_initial_window_withReplies.parquet` into a DataFrame (`df`).

#### 2️⃣ Data Integrity
- Ensures required columns exist (`show_key`, `text`, `like_count`, `published_at`, etc.).
- Converts `published_at` and `upload_date` to datetime objects.
- Fills missing `upload_date` values using the earliest comment timestamp for each video.

#### 3️⃣ Text Cleaning
- Removes zero-width spaces, extra whitespace, and URLs.
- Saves the cleaned text back to `df["text"]`.

#### 4️⃣ Feature Engineering
Adds text-based features for downstream analysis:
- `text_len` → number of characters  
- `word_count` → number of words  
- `has_url` → flag if a URL is present  
- `num_emojis`, `num_caps`, `num_exclaims` → emoji count, uppercase letters, exclamation marks

#### 5️⃣ Spam Flagging
Uses lightweight regex rules to identify probable spam:
- Very short or empty comments  
- Promotional keywords (`promo`, `telegram`, `whatsapp`, `crypto`, `giveaway`, etc.)  
- Excessive character repetition (e.g., “loooool”)  

Creates a boolean column `spam_flag`.

#### 6️⃣ Language Filtering
- Detects comment language with `langdetect`.  
- Keeps only English (`en`, `en-us`, `en-gb`).

#### 7️⃣ Deduplication
- Builds lowercase, punctuation-free **fingerprints** of each comment.  
- Removes duplicates within the same video (exact + near duplicates).

#### 8️⃣ Temporal Window & Phase Labeling
- Computes `days_since_upload = published_at – upload_date`.  
- Keeps only comments posted within **0–180 days (6 months)**.  
- Assigns time phases:  
  - `live` (0–1 day)  
  - `early` (2–7 days)  
  - `mid` (8–30 days)  
  - `late` (31–180 days)  
- Adds `era` = “before” or “after” based on year (< 2020 / ≥ 2020).

#### 9️⃣ Engagement Metrics
- Converts `like_count` and `reply_count` to numeric.  
- Combines them into `eng_score` = `like_count` + `reply_count`.

#### 🔟 Balancing & Weights
- Counts comments per show.  
- If all shows have data, downsamples each to the smallest group → **balanced subset**.  
- Adds `w_show_equal` so each show can be equally weighted in later analysis.

#### 11️⃣ Column Selection & Saving
- Keeps relevant columns: metadata, features, flags, engagement, phase, and weights.  
- Saves cleaned datasets and a summary manifest.

---

### 📁 Outputs

| File | Purpose |
|------|----------|
| **`comments_yt_180d_clean_unbalanced.parquet` / `.csv`** | Main cleaned dataset |
| **`comments_yt_180d_clean_balanced.parquet` / `.csv`** | Equal-sized sample per show (if balancing possible) |
| **`preprocess_manifest_counts.csv`** | Count summary by show × kind × phase |

**Total files:** 3 – 5, depending on whether balancing succeeded.

---

### 🎯 Result
After running this code, you have a fully processed dataset that is:
- Cleaned, deduplicated, and language-filtered  
- Labeled by **show**, **era**, and **time phase**  
- Ready for:
  - VADER sentiment and EDA (Mikyung)  
  - Topic modeling & temporal trends (Emma)  
  - Statistical tests and visualizations (Julie)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install and Load

In [ ]:
!pip install -q pandas pyarrow langdetect emoji

### Youtube Cleaning + Processing

In [ ]:
import pandas as pd
import numpy as np
import re, math
from datetime import timedelta
import emoji

# --------- paths ---------
IN_PATH  = "/content/drive/MyDrive/unstructured_assignments/final_project/comments_initial_window_withReplies.parquet"  # YouTube combined (180d, top+replies)
OUT_DIR  = "/content/drive/MyDrive/unstructured_assignments/final_project/"

df = pd.read_parquet(IN_PATH)

# Ensure key fields exist
for c in ["platform","video_id","show_key","show_year","comment_id","parent_id","text","published_at","upload_date","like_count","reply_count","kind"]:
    if c not in df.columns:
        df[c] = None

# Stamp platform
df["platform"] = df.get("platform").fillna("youtube")

# Parse timestamps
df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
df["upload_date"]  = pd.to_datetime(df["upload_date"],  errors="coerce")

# Keep rows with usable times
df = df[df["published_at"].notna()].copy()
# If upload_date missing (shouldn’t happen with your scrape), proxy with min comment time per video
if df["upload_date"].isna().any():
    proxy = df.groupby("video_id")["published_at"].min().rename("upload_date_proxy")
    df = df.merge(proxy, on="video_id", how="left")
    df["upload_date"] = df["upload_date"].fillna(df["upload_date_proxy"])
    df = df.drop(columns=["upload_date_proxy"])


/tmp/ipython-input-288622989.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["upload_date"] = df["upload_date"].fillna(df["upload_date_proxy"])


In [ ]:
# === CLEANING & PREPROCESSING ===
!pip install -q pandas pyarrow langdetect emoji

import pandas as pd
import numpy as np
import re, math
from datetime import timedelta
import emoji
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42

# --------- paths ---------
IN_PATH  = "/content/drive/MyDrive/unstructured_assignments/final_project/comments_initial_window_withReplies.parquet"
OUT_DIR  = "/content/drive/MyDrive/unstructured_assignments/final_project/"

df = pd.read_parquet(IN_PATH)

# Ensure key fields exist
for c in ["platform","video_id","show_key","show_year","comment_id","parent_id","text",
          "published_at","upload_date","like_count","reply_count","kind"]:
    if c not in df.columns:
        df[c] = None

df["platform"] = df.get("platform").fillna("youtube")

# --- timestamps ---
df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
df["upload_date"]  = pd.to_datetime(df["upload_date"],  errors="coerce")
df = df[df["published_at"].notna()].copy()
if df["upload_date"].isna().any():
    proxy = df.groupby("video_id")["published_at"].min().rename("upload_date_proxy")
    df = df.merge(proxy, on="video_id", how="left")
    df["upload_date"] = df["upload_date"].fillna(df["upload_date_proxy"])
    df = df.drop(columns=["upload_date_proxy"])

# ---------- text cleaning ----------
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)

def clean_text(t: str) -> str:
    if not isinstance(t, str): return ""
    t = t.replace("\u200b", "").strip()
    t = re.sub(r"\s+", " ", t)
    t = URL_RE.sub("", t)
    return t

df["text"] = df["text"].fillna("").apply(clean_text)

# ---------- derived features ----------
def count_emojis(s: str) -> int:
    return sum(ch in emoji.EMOJI_DATA for ch in s)

df["text_len"]     = df["text"].str.len()
df["word_count"]   = df["text"].str.split().apply(len)
df["has_url"]      = df["text"].str.contains(r"(?:http|www\.)", case=False, regex=True, na=False)
df["num_emojis"]   = df["text"].apply(count_emojis)
df["num_caps"]     = df["text"].str.findall(r"[A-Z]").str.len()
df["num_exclaims"] = df["text"].str.count("!")

# ---------- spam flag ----------
promo_terms = r"(?:\bpromo\b|\btelegram\b|\bwhatsapp\b|\bdm me\b|\bfollow me\b|\bfree\s*\$\b|\bcrypto\b|\bgiveaway\b)"
df["spam_flag"] = (
    (df["text_len"] < 3) |
    (df["word_count"] == 0) |
    df["text"].str.contains(promo_terms, case=False, regex=True, na=False)
)

REPEAT_RE = re.compile(r"(.)\1{6,}")
df["spam_flag"] = df["spam_flag"] | df["text"].str.contains(REPEAT_RE, na=False)

# ---------- language filter (English) ----------
def detect_lang_safe(x):
    try: return detect(x)
    except: return "unk"

df["lang"] = df["text"].apply(detect_lang_safe)
keep_langs = {"en","en-us","en-gb"}
if (df["lang"].isin(keep_langs)).sum() > 0:
    df = df[df["lang"].isin(keep_langs)].copy()

# ---------- dedupe ----------
def fingerprint(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["fp"] = df["text"].apply(fingerprint)
df = df.drop_duplicates(subset=["platform","video_id","text"])
df = df.drop_duplicates(subset=["platform","video_id","fp"])

# ---------- temporal features ----------
df["days_since_upload"] = (df["published_at"] - df["upload_date"]).dt.days
df = df[(df["days_since_upload"] >= 0) & (df["days_since_upload"] <= 180)].copy()

bins  = [-0.1, 1, 7, 30, 180]
labels= ["live","early","mid","late"]
df["phase"] = pd.cut(df["days_since_upload"], bins=bins, labels=labels)

df["era"] = np.where(pd.to_numeric(df["show_year"], errors="coerce") >= 2020, "after", "before")

df["like_count"]  = pd.to_numeric(df["like_count"], errors="coerce")
df["reply_count"] = pd.to_numeric(df["reply_count"], errors="coerce")
df["eng_score"]   = df[["like_count","reply_count"]].fillna(0).sum(axis=1)

# ---------- balance & weights ----------
counts = df.groupby("show_key").size().sort_values()
can_balance = (len(counts) > 0 and counts.min() > 0)
if can_balance:
    target = int(counts.min())
    balanced = (df.groupby("show_key", group_keys=False, sort=False)
                  .sample(n=target, random_state=42)
                  .reset_index(drop=True))
else:
    balanced = pd.DataFrame(columns=df.columns)

# equal weights for all shows
show_counts = df["show_key"].value_counts()
df["w_show_equal"] = df["show_key"].map(lambda k: show_counts.min() / show_counts[k])
if "balanced" in locals() and not balanced.empty:
    balanced = balanced.copy()
    balanced["w_show_equal"] = 1.0

# ---------- final columns ----------
cols_keep = [
    "platform","show_key","show_year","era","video_id","comment_id","parent_id","kind",
    "text","text_len","word_count","has_url","num_emojis","num_caps","num_exclaims","spam_flag",
    "like_count","reply_count","eng_score","published_at","upload_date","days_since_upload","phase",
    "w_show_equal"
]
for c in cols_keep:
    if c not in df.columns: df[c] = None
    if "balanced" in locals() and not balanced.empty and c not in balanced.columns:
        balanced[c] = None

df_out = df[cols_keep].copy()
if "balanced" in locals() and not balanced.empty:
    balanced_out = balanced[cols_keep].copy()

# ---------- save ----------
df_out.to_parquet(f"{OUT_DIR}/comments_yt_180d_clean_unbalanced.parquet", index=False)
df_out.to_csv    (f"{OUT_DIR}/comments_yt_180d_clean_unbalanced.csv", index=False)

if "balanced" in locals() and not balanced.empty:
    balanced_out.to_parquet(f"{OUT_DIR}/comments_yt_180d_clean_balanced.parquet", index=False)
    balanced_out.to_csv    (f"{OUT_DIR}/comments_yt_180d_clean_balanced.csv", index=False)

# ---------- profile summary ----------
profile = (df_out.groupby(["show_key","kind","phase"]).size()
           .rename("n").reset_index()
           .sort_values(["show_key","kind","phase"]))
profile.to_csv(f"{OUT_DIR}/preprocess_manifest_counts.csv", index=False)

summary = {
    "total_rows_unbalanced": int(len(df_out)),
    "per_show_counts_unbalanced": df_out["show_key"].value_counts().to_dict(),
    "balanced_available": bool(len(balanced) > 0),
    "per_show_counts_balanced": (balanced_out["show_key"].value_counts().to_dict()
                                 if not balanced.empty else {}),
}
summary


/tmp/ipython-input-2732136061.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["upload_date"] = df["upload_date"].fillna(df["upload_date_proxy"])
/tmp/ipython-input-2732136061.py:68: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["spam_flag"] = df["spam_flag"] | df["text"].str.contains(REPEAT_RE, na=False)
/tmp/ipython-input-2732136061.py:148: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  profile = (df_out.groupby(["show_key","kind","phase"]).size()


{'total_rows_unbalanced': 15336,
 'per_show_counts_unbalanced': {'2017_Lady_Gaga': 3681,
  '2022_Dre_Snoop_Eminem_MaryJ_Kendrick': 3081,
  '2020_Shakira_JLo': 2791,
  '2016_Coldplay_Beyonce_Bruno': 2483,
  '2023_Rihanna': 2346,
  '2015_Katy_Perry': 954},
 'balanced_available': True,
 'per_show_counts_balanced': {'2015_Katy_Perry': 954,
  '2016_Coldplay_Beyonce_Bruno': 954,
  '2017_Lady_Gaga': 954,
  '2020_Shakira_JLo': 954,
  '2022_Dre_Snoop_Eminem_MaryJ_Kendrick': 954,
  '2023_Rihanna': 954}}

### Peak in the Data

In [ ]:
import pandas as pd
import os

# List of expected output files
files = [
    "/content/drive/MyDrive/unstructured_assignments/final_project/comments_yt_180d_clean_unbalanced.parquet",
    "/content/drive/MyDrive/unstructured_assignments/final_project/comments_yt_180d_clean_balanced.parquet",
    "/content/drive/MyDrive/unstructured_assignments/final_project/comments_yt_180d_clean_unbalanced.csv",
    "/content/drive/MyDrive/unstructured_assignments/final_project/comments_yt_180d_clean_balanced.csv",
    "/content/drive/MyDrive/unstructured_assignments/final_project/preprocess_manifest_counts.csv",
]

# Loop through and preview
for f in files:
    if os.path.exists(f):
        print(f"\n=== {f} ===")
        try:
            df = pd.read_parquet(f) if f.endswith(".parquet") else pd.read_csv(f)
            print(f"Shape: {df.shape}")
            display(df.head(3),   # shows first few rows
            df.columns.tolist(),
            df["show_key"].value_counts(),
            df["phase"].value_counts()
            )
        except Exception as e:
            print(f"Error reading {f}: {e}")
    else:
        print(f"{f} not found.")



=== /content/drive/MyDrive/unstructured_assignments/final_project/comments_yt_180d_clean_unbalanced.parquet ===
Shape: (15336, 24)


,platform,show_key,show_year,era,video_id,comment_id,parent_id,kind,text,text_len,...,num_exclaims,spam_flag,like_count,reply_count,eng_score,published_at,upload_date,days_since_upload,phase,w_show_equal
0,youtube,2015_Katy_Perry,2015,before,ZD1QrIe--_Y,UgiHJoeICH27JXgCoAEC,None,top_level,Left Shark... Please..,22,...,0,False,0,0.0,0.0,2017-03-13 04:32:55+00:00,2016-09-13 19:01:53+00:00,180,late,1.0
1,youtube,2015_Katy_Perry,2015,before,ZD1QrIe--_Y,Ughs3y4nmV1EAHgCoAEC,None,top_level,She's bae 😍😍😍,13,...,0,False,0,0.0,0.0,2017-03-13 03:15:56+00:00,2016-09-13 19:01:53+00:00,180,late,1.0
2,youtube,2015_Katy_Perry,2015,before,ZD1QrIe--_Y,UgjceiGZVJW0angCoAEC,None,top_level,"I don't care what anyone says, this is my favo...",69,...,0,False,0,0.0,0.0,2017-03-13 02:32:38+00:00,2016-09-13 19:01:53+00:00,180,late,1.0


['platform',
 'show_key',
 'show_year',
 'era',
 'video_id',
 'comment_id',
 'parent_id',
 'kind',
 'text',
 'text_len',
 'word_count',
 'has_url',
 'num_emojis',
 'num_caps',
 'num_exclaims',
 'spam_flag',
 'like_count',
 'reply_count',
 'eng_score',
 'published_at',
 'upload_date',
 'days_since_upload',
 'phase',
 'w_show_equal']

,count
show_key,
2017_Lady_Gaga,3681
2022_Dre_Snoop_Eminem_MaryJ_Kendrick,3081
2020_Shakira_JLo,2791
2016_Coldplay_Beyonce_Bruno,2483
2023_Rihanna,2346
2015_Katy_Perry,954


,count
phase,
late,7142
mid,4416
early,2295
live,1483


...


=== /content/drive/MyDrive/unstructured_assignments/final_project/preprocess_manifest_counts.csv ===
Shape: (48, 4)


,show_key,kind,phase,n
0,2015_Katy_Perry,reply,live,5
1,2015_Katy_Perry,reply,early,0
2,2015_Katy_Perry,reply,mid,2


['show_key', 'kind', 'phase', 'n']

,count
show_key,
2015_Katy_Perry,8
2016_Coldplay_Beyonce_Bruno,8
2017_Lady_Gaga,8
2020_Shakira_JLo,8
2022_Dre_Snoop_Eminem_MaryJ_Kendrick,8
2023_Rihanna,8


,count
phase,
live,12
early,12
mid,12
late,12


## For Reddit

### Methodology

#### Data Source and Objective
This portion of the study focuses on audience reactions to **Super Bowl Halftime Shows** from Reddit discussions, complementing the YouTube comment dataset.  
Comments were collected from public Reddit threads corresponding to six official halftime shows:  
three before the NFL–Jay-Z partnership (2015 Katy Perry, 2016 Coldplay ft. Beyoncé & Bruno Mars, 2017 Lady Gaga) and three after (2020 Shakira & Jennifer Lopez, 2022 Dr. Dre & ensemble, 2023 Rihanna).  
All data were publicly available and obtained using Pushshift and Reddit’s API through custom Python scripts.

---

#### Environment and Input
Data processing was performed in **Google Colab** using Python 3 with the libraries `pandas`, `numpy`, and `pyarrow`.  
The combined dataset `halftime_comments_all.csv` contained both **thread-level** and **comment-level** information, including:  
`submission_id`, `comment_id`, `author`, `score`, `body`, `created_utc`, `thread_title`, `thread_subreddit`, `year`, `artists`, and timestamp fields.

---

#### Cleaning Procedure
1. **Removed invalid or bot content**  
   Deleted comments, moderator removals, and bot accounts were excluded (`[deleted]`, `[removed]`, or usernames containing “bot”).  

2. **Text normalization**  
   - Collapsed multiple spaces and removed URLs.  
   - Trimmed leading/trailing whitespace.  
   - Filtered out very short or empty comments.  

3. **Spam filtering**  
   - Comments containing promotional or repetitive text (e.g., “buy”, “follow”, “discount”, “giveaway”, or long character repeats) were flagged and removed.  

4. **Variable standardization**  
   To align with the YouTube dataset, new columns were added:  
   - `platform` = `"reddit"`  
   - `show_year` = year of performance  
   - `show_key` = `"{year}_{artist}"`  
   - `created_dt` = parsed comment datetime  
   - `reply_depth` = 0 for top-level comments, 1 for replies  

5. **6-month window filter**  
   Only comments posted within **180 days** after each thread’s creation date (`thread_created_iso`) were kept, capturing the **initial reception** period following each halftime performance.  

6. **Engagement features**  
   - `score` (upvotes) retained as numeric  
   - `reply_depth` used to distinguish direct discussion vs. replies  

7. **Column pruning and deduplication**  
   Retained key analytical fields (`platform`, `show_key`, `show_year`, `author`, `text`, `created_dt`, `score`, `reply_depth`, `thread_subreddit`, `artists`, `year`) and dropped duplicates.

---

#### Balancing and Output
To ensure comparability across shows, datasets were balanced by randomly sampling each show to the same number of comments (`n = min_count`).  
Four output files were generated for consistency with the YouTube workflow:

| Type | Format | Description |
|------|---------|-------------|
| Unbalanced | `.csv` & `.parquet` | All valid comments within 6 months |
| Balanced | `.csv` & `.parquet` | Equalized number of comments per show |

Example outputs:
- `data/comments_reddit_180d_clean_unbalanced.parquet`  
- `data/comments_reddit_180d_clean_balanced.parquet`

---

#### Quality Control
- Verified that all `show_key` values corresponded to one of the six halftime shows.  
- Confirmed no missing timestamps after filtering.  
- Summarized per-show counts before and after balancing for documentation.  
- Ensured text integrity (no encoding artifacts, emojis preserved).

---

#### Next Analytical Steps
The cleaned Reddit dataset now supports:
1. **Sentiment analysis** using VADER for polarity and intensity.  
2. **Topic modeling** with BERTopic or LDA to uncover discussion themes.  
3. **Cross-platform comparison** (Reddit vs YouTube) of sentiment, engagement, and topic prevalence before and after the NFL–Jay-Z partnership.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Import

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/unstructured_assignments/final_project/halftime_comments_all.csv")


### Reddit Cleaning + Processing

In [ ]:
# --- Setup ---
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# Paths
IN_FILE = "/content/drive/MyDrive/unstructured_assignments/final_project/halftime_comments_all.csv"
OUT_DIR = "/content/drive/MyDrive/unstructured_assignments/final_project/"
os.makedirs(OUT_DIR, exist_ok=True)

# --- Load ---
df = pd.read_csv(IN_FILE)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

# --- Basic cleaning ---
# Remove deleted or bot comments
mask_valid = (
    df["body"].notna()
    & (~df["body"].str.lower().isin(["[deleted]", "[removed]"]))
    & (~df["author"].str.contains("bot", case=False, na=False))
)
df = df[mask_valid].copy()

# Clean text
df["text"] = (
    df["body"]
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"http\S+|www\S+", "", regex=True)
    .str.strip()
)
df = df[df["text"].str.len() > 5].copy()

# --- Create unified fields ---
df["platform"] = "reddit"
df["show_year"] = df["year"]
df["show_key"] = df["year"].astype(str) + "_" + df["artists"].str.replace(" ", "_").str[:25]
df["created_dt"] = pd.to_datetime(df["comment_created_iso"], errors="coerce")

# --- 6-month window filter ---
# Convert thread_created_iso to datetime
df["thread_created_dt"] = pd.to_datetime(df["thread_created_iso"], errors="coerce")
df["delta_days"] = (df["created_dt"] - df["thread_created_dt"]).dt.days

mask_window = (df["delta_days"].between(0, 180))  # within 6 months of post
df = df[mask_window].copy()
print("Rows after 6-month window:", len(df))

# --- Engagement features ---
df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0)
df["reply_depth"] = np.where(df["parent_id"].str.startswith("t1_"), 1, 0)  # 0=top-level, 1=reply

# --- Light spam filtering ---
promo_terms = r"(buy|discount|subscribe|follow|merch|promo|sale|giveaway)"
df["is_spam"] = (
    df["text"].str.contains(promo_terms, case=False, regex=True)
    | df["text"].str.contains(r"(.)\1{6,}", regex=True)  # long letter spam
)
df = df[~df["is_spam"]].copy()

# --- Drop unneeded columns ---
cols_keep = [
    "platform", "show_key", "show_year",
    "author", "text", "created_dt", "score",
    "reply_depth", "thread_subreddit", "artists", "year"
]
df_out = df[cols_keep].drop_duplicates(subset=["author", "text"]).reset_index(drop=True)

# --- Balance across shows ---
counts = df_out["show_key"].value_counts()
min_count = counts.min()
print("Counts per show before balancing:\n", counts)
print("Balancing target per show:", min_count)

balanced = (
    df_out.groupby("show_key", group_keys=False)
    .apply(lambda g: g.sample(n=min_count, random_state=42))
    .reset_index(drop=True)
)
print("Balanced shape:", balanced.shape)

# --- Save outputs ---
unbalanced_path_parquet = os.path.join(OUT_DIR, "comments_reddit_180d_clean_unbalanced.parquet")
balanced_path_parquet = os.path.join(OUT_DIR, "comments_reddit_180d_clean_balanced.parquet")
unbalanced_path_csv = os.path.join(OUT_DIR, "comments_reddit_180d_clean_unbalanced.csv")
balanced_path_csv = os.path.join(OUT_DIR, "comments_reddit_180d_clean_balanced.csv")

df_out.to_parquet(unbalanced_path_parquet, index=False)
balanced.to_parquet(balanced_path_parquet, index=False)
df_out.to_csv(unbalanced_path_csv, index=False)
balanced.to_csv(balanced_path_csv, index=False)

print("\n✅ Saved cleaned Reddit data:")
print(f"- {unbalanced_path_parquet}")
print(f"- {balanced_path_parquet}")
print(f"- {unbalanced_path_csv}")
print(f"- {balanced_path_csv}")

# --- Summary ---
print("\nCounts per show (balanced):")
print(balanced["show_key"].value_counts())


Raw shape: (27680, 18)
Columns: ['submission_id', 'comment_id', 'author', 'score', 'body', 'created_utc', 'parent_id', 'thread_title', 'thread_url', 'thread_subreddit', 'thread_author', 'thread_created_utc', 'thread_num_comments', 'thread_post_score', 'year', 'artists', 'comment_created_iso', 'thread_created_iso']
Rows after 6-month window: 26121


/tmp/ipython-input-1950022490.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["text"].str.contains(promo_terms, case=False, regex=True)
/tmp/ipython-input-1950022490.py:58: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | df["text"].str.contains(r"(.)\1{6,}", regex=True)  # long letter spam
/tmp/ipython-input-1950022490.py:78: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min_count, random_state=42))


Counts per show before balancing:
 show_key
2022_Dr._Dre,_Snoop_Dogg,_Emin    9165
2023_Rihanna                      6387
2017_Lady_Gaga                    3961
2020_Shakira,_Jennifer_Lopez      3575
2016_Coldplay,_Beyoncé,_Bruno_    2136
2015_Katy_Perry                    360
Name: count, dtype: int64
Balancing target per show: 360
Balanced shape: (2160, 11)

✅ Saved cleaned Reddit data:
- /content/drive/MyDrive/unstructured_assignments/final_project/comments_reddit_180d_clean_unbalanced.parquet
- /content/drive/MyDrive/unstructured_assignments/final_project/comments_reddit_180d_clean_balanced.parquet
- /content/drive/MyDrive/unstructured_assignments/final_project/comments_reddit_180d_clean_unbalanced.csv
- /content/drive/MyDrive/unstructured_assignments/final_project/comments_reddit_180d_clean_balanced.csv

Counts per show (balanced):
show_key
2015_Katy_Perry                   360
2016_Coldplay,_Beyoncé,_Bruno_    360
2017_Lady_Gaga                    360
2020_Shakira,_Jennifer_Lopez